# Data Leakage in den Ranglistenpositionen und Team Ranking

- Rangliste = Zustand Ende der Saison 
- somit Grand Tours bereits mit eingepreist
- Lösung: Den Vorjahreswert als Ausgangspunkt in die Modelle bringen

- dazu Scraper 08...py, um die Werte für 2004 zu bekommen
    - joinen mit unserem Datensatz inkl lagging -> Werte 2006 sind bspw. Startwert für 2007
    - dazu neue lag_ spalten erstellen für Sauberkeit
    
- Team Index neu berechnen (vgl. Notebook 07-11)
- Test Missing Values

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

In [31]:
pickle_path = '../../data/processed/25_cleaned_master_data.pkl'

df = pd.read_pickle(pickle_path)

In [32]:
print(df.columns.tolist())

['meta_race', 'meta_year', 'meta_url', 'rank', 'meta_rider_url', 'height', 'meta_name', 'meta_nationality', 'weight', 'meta_url_name', 'meta_departure', 'meta_arrival', 'distance', 'vertical_meters', 'one_day_races', 'gc', 'time_trial', 'sprint', 'climber', 'hills', 'stage_nr', 'meta_date', 'meta_departure_lat', 'meta_departure_lon', 'meta_arrival_lat', 'meta_arrival_lon', 'rider_points_season', 'rider_rank_season', 'meta_current_team', 'team_tier', 'age_at_race', 'rider_bmi', 'race_competitiveness_median', 'team_power_index', 'wind_stability_index', 'weather_temp_mean', 'weather_temp_trend', 'weather_rain_prob_mean', 'weather_precipitation_mean', 'weather_humidity_mean', 'won_how_cat', 'gradient_final_km']


In [34]:
# Laden der Jsonl aus dem Scraper 08



rank_history_df = pd.read_json(
    "../../data/raw/rider_rank_history.jsonl",
    lines=True
)

print(rank_history_df.shape)

rank_history_df.head()

(19375, 4)


,url_name,season,points,rank
0,tom-boonen,2017,224,294
1,tom-boonen,2016,925,36
2,tom-boonen,2015,962,35
3,tom-boonen,2014,767,45
4,tom-boonen,2013,137,413


# Daten "Laggen" 'rider_points_season', 'rider_rank_season' 
- damit die Daten aus 2005 nun als Basis für 2006 dienen bspw.

In [35]:
# Zwischen df erzeugen zur Vorbereitung

lag_df = rank_history_df.copy()

lag_df = lag_df.rename(
    columns={
        "url_name": "meta_rider_url",
        "points": "lag_rider_points_season",
        "rank": "lag_rider_rank_season"
    }
)

In [ ]:
# Season Wert auf das Folgejahr verschieben

lag_df["meta_year"] = lag_df["season"] + 1


In [40]:
lag_df[
    ["meta_rider_url", "meta_year"]
].duplicated().sum()

np.int64(0)

In [ ]:
lag_df.head()


,meta_rider_url,season,lag_rider_points_season,lag_rider_rank_season,meta_year
0,tom-boonen,2017,224,294,2018
1,tom-boonen,2016,925,36,2017
2,tom-boonen,2015,962,35,2016
3,tom-boonen,2014,767,45,2015
4,tom-boonen,2013,137,413,2014


### Joinen des Hilfs- und Hauptdatensatzes

In [ ]:
df = df.merge(
    lag_df[
        [
            "meta_rider_url",
            "meta_year",
            "lag_rider_points_season",
            "lag_rider_rank_season"
        ]
    ],
    on=[
        "meta_rider_url",
        "meta_year"
    ],
    how="left"
)

### Erfolgstest

In [41]:
sample_rows = (
    df[
        [
            "meta_rider_url",
            "meta_year",
            "lag_rider_points_season",
            "lag_rider_rank_season"
        ]
    ]
    .drop_duplicates()
    .dropna(subset=[
        "lag_rider_points_season",
        "lag_rider_rank_season"
    ])
    .sample(3, random_state=42)
)

sample_rows

,meta_rider_url,meta_year,lag_rider_points_season,lag_rider_rank_season
108104,sergio-luis-henao,2018,860.0,52.0
18549,thomas-voeckler,2011,741.0,44.0
156015,winner-anacona,2013,148.0,418.0


# NA Test

In [43]:
print(df["lag_rider_points_season"].isna().sum())
print(df["lag_rider_rank_season"].isna().sum())

2391
2391


In [44]:
missing_riders = (
    df.loc[
        df["lag_rider_points_season"].isna(),
        "meta_rider_url"
    ]
    .unique()
)

print(f"Betroffene Fahrer: {len(missing_riders)}")

Betroffene Fahrer: 119


In [47]:
missing_df = (
    df.loc[
        df["lag_rider_points_season"].isna(),
        [
            "meta_rider_url",
            "meta_year",
            "rider_points_season",
            "rider_rank_season",
            "lag_rider_points_season",
            "lag_rider_rank_season"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["meta_rider_url", "meta_year"]
    )
)

print(missing_df.shape)

missing_df.head()

missing_df.to_csv(
    "../../data/interim/missing_lag_rider_values.csv",
    index=False
)

(134, 6)


### Bereinigung der NAs
- Fahrer in diesen Jahren nicht in der Rangliste geführt
- somit wie in 07-05 Points = 0 und Rank = 9999

In [48]:
df["lag_rider_rank_season"] = (
    df["lag_rider_rank_season"]
    .fillna(9999)
)

df["lag_rider_points_season"] = (
    df["lag_rider_points_season"]
    .fillna(0)
)

In [49]:
print(df["lag_rider_points_season"].isna().sum())
print(df["lag_rider_rank_season"].isna().sum())

0
0


# Fahrerfeld Index neu berechnen (vgl. 07-05)
- Basis Lag-Werte
- 'lag_race_competitiveness_median'

In [51]:
etappen_gruppen = df.groupby(
    ["meta_race", "meta_year", "stage_nr"]
)

df["lag_race_competitiveness_median"] = (
    etappen_gruppen["lag_rider_rank_season"]
    .transform("median")
)

C:\Users\lukas\AppData\Local\Temp\ipykernel_18756\220104072.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  etappen_gruppen = df.groupby(


# Team Index neu berechnen (vgl. 07-05)
- Basis Lag-Werte
- 'lag_team_power_index'

In [54]:
etappen_team_gruppen = df.groupby(
    ["meta_race", "meta_year", "stage_nr", "meta_current_team"]
)

df["lag_team_power_index"] = (
    etappen_team_gruppen["lag_rider_rank_season"]
    .transform("median")
)

C:\Users\lukas\AppData\Local\Temp\ipykernel_18756\228506047.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  etappen_team_gruppen = df.groupby(


In [55]:
df.isna().sum()

meta_race                          0
meta_year                          0
meta_url                           0
rank                               0
meta_rider_url                     0
height                             0
meta_name                          0
meta_nationality                   0
weight                             0
meta_url_name                      0
meta_departure                     0
meta_arrival                       0
distance                           0
vertical_meters                    0
one_day_races                      0
gc                                 0
time_trial                         0
sprint                             0
climber                            0
hills                              0
stage_nr                           0
meta_date                          0
meta_departure_lat                 0
meta_departure_lon                 0
meta_arrival_lat                   0
meta_arrival_lon                   0
rider_points_season                0
r

# Ursprünglichen Ranking Spalten als meta_kennzeichenn

In [57]:
df = df.rename(
    columns={
        "rider_points_season": "meta_rider_points_season",
        "rider_rank_season": "meta_rider_rank_season",
        "race_competitiveness_median": "meta_race_competitiveness_median",
        "team_power_index": "meta_team_power_index",
    }
)

In [58]:
[
    col
    for col in df.columns
    if "rank" in col.lower()
    or "point" in col.lower()
    or "competitiveness" in col.lower()
    or "power" in col.lower()
]

['rank',
 'meta_rider_points_season',
 'meta_rider_rank_season',
 'meta_race_competitiveness_median',
 'meta_team_power_index',
 'lag_rider_points_season',
 'lag_rider_rank_season',
 'lag_race_competitiveness_median',
 'lag_team_power_index']

# Checkpointing



In [59]:
pfad = '../../data/processed/26_cleaned_master_data.pkl'
df.to_pickle(pfad)